# MLP baseline: random search, then the winner x 5 seeds

Runs on a Kaggle GPU notebook: a plain neural net on the same 93 inputs as LightGBM, no message passing (`conf/hyperparameters/mlp/space.yaml`). About 10 hours (the sister repository's search took 9 h on a T4, a final seed about 5 min).
- Settings (right panel): Accelerator = GPU, Internet = On.
- Input: the dataset `sandrokhizanishvili/hi-small-gnn`.
- Run All (or Save Version -> Save & Run All to run headless, 12 h limit).
- The search writes `trials.csv`, `configs.json`, `recipe.json` to `/kaggle/working/outputs/tuning/mlp/mlp_readout-full/`; the final retrain writes `results.json`, `history.csv`, `curves.png`, `predictions.csv`, `best.pt` to `/kaggle/working/outputs/final/mlp/mlp_readout-full/seed-<s>/`.

In [1]:
# The pinned torch / PyG stack known to work on Kaggle GPU; do not change versions.
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric lightgbm==4.6.0 hydra-core==1.3.2

Found existing installation: torch 2.11.0+cu130
Uninstalling torch-2.11.0+cu130:
  Successfully uninstalled torch-2.11.0+cu130
Found existing installation: torchvision 0.26.0+cu130
Uninstalling torchvision-0.26.0+cu130:
  Successfully uninstalled torchvision-0.26.0+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 229.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 449.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 262.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 258.3 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 278.6 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 264.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [2]:
# The code: clone once, pull on a rerun.
import os
if os.path.isdir('/kaggle/working/gnn_for_aml'):
    !cd /kaggle/working/gnn_for_aml && git pull
else:
    !git clone https://github.com/sandrokhizanishvili/gnn_for_aml.git /kaggle/working/gnn_for_aml
%cd /kaggle/working/gnn_for_aml

Cloning into '/kaggle/working/gnn_for_aml'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 189 (delta 83), reused 156 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (189/189), 362.28 KiB | 8.84 MiB/s, done.
Resolving deltas: 100% (83/83), done.
/kaggle/working/gnn_for_aml


In [3]:
# Environment test: versions, the GPU.
!python run.py mode=check machine=kaggle

python 3.13.16 | torch 2.8.0+cu126 | CUDA build 12.6 | device cuda
  torch_geometric 2.8.1
  torch_scatter 2.1.2+pt28cu126
  torch_sparse 0.6.18+pt28cu126
  lightgbm 4.6.0
  hydra-core 1.3.2
  hydra-submitit-launcher not installed (only needed on Cineca)
  numpy 2.1.3
  scipy 1.16.3
  pandas 2.3.3
  scikit-learn 1.6.1
  matplotlib 3.10.0
  psutil 5.9.5
/usr/local/lib/python3.13/dist-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(
  gin         params   54,083 | loss 0.7224
  pna         params  176,833 | loss 0.6495
  gat         params   53,953 | loss 0.7796
  transformer params   70,337 | loss 0.6803
OK


In [4]:
# The search: 100 trials, seed 42, ranked by validation PR-AUC -> recipe.json. Finished trials are kept in trials.csv, so the cell can be rerun after an interruption.
!python run.py -m tuning=mlp/search machine=kaggle

[2026-11-14 17:54:03,696][HYDRA] Launching 1 jobs locally
[2026-11-14 17:54:03,696][HYDRA] 	#0 : tuning=mlp/search machine=kaggle
=== tune mlp_readout-full seed 42 -> /kaggle/working/outputs/tuning/mlp/mlp_readout-full
mode: tune
seed: 42
trial: null
node_encoding: none
gfp_sheet: default
hyperparameters:
  for: mlp
  epochs: 60
  patience: 10
  std_floor: 0.001
  search:
    n_trials: 100
    seeds:
    - 42
    select: val_pr_auc
    draw_seed: 42
    start:
      hidden: 512
      depth: 3
      shape: funnel
      norm: batch
      dropout: 0.2
      lr: 0.001
      weight_decay: 0.0001
      pos_weight: 8.0
      batch_size: 8192
    space:
      hidden:
      - 128
      - 256
      - 512
      - 1024
      depth:
      - 2
      - 3
      - 4
      - 5
      shape:
      - flat
      - funnel
      - resnet
      norm:
      - none
      - batch
      dropout:
      - 0.1
      - 0.2
      - 0.3
      - 0.5
      lr:
      - 0.0003
      - 0.001
      - 0.003
      weight_decay:

In [5]:
# The final retrain: the winner of recipe.json x seeds 42-46. A finished seed is skipped.
!python run.py -m tuning=mlp/final machine=kaggle

[2026-11-15 02:46:13,739][HYDRA] Launching 5 jobs locally
[2026-11-15 02:46:13,739][HYDRA] 	#0 : seed=42 tuning=mlp/final machine=kaggle
=== final mlp_readout-full seed 42 -> /kaggle/working/outputs/final/mlp/mlp_readout-full/seed-42
mode: final
seed: 42
trial: null
node_encoding: none
gfp_sheet: default
hyperparameters:
  for: mlp
  epochs: 60
  patience: 10
  std_floor: 0.001
  search:
    n_trials: 100
    seeds:
    - 42
    select: val_pr_auc
    draw_seed: 42
    start:
      hidden: 512
      depth: 3
      shape: funnel
      norm: batch
      dropout: 0.2
      lr: 0.001
      weight_decay: 0.0001
      pos_weight: 8.0
      batch_size: 8192
    space:
      hidden:
      - 128
      - 256
      - 512
      - 1024
      depth:
      - 2
      - 3
      - 4
      - 5
      shape:
      - flat
      - funnel
      - resnet
      norm:
      - none
      - batch
      dropout:
      - 0.1
      - 0.2
      - 0.3
      - 0.5
      lr:
      - 0.0003
      - 0.001
      - 0.003
   

In [6]:
# The metrics of every finished seed, one row each.
import glob, json, pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob('/kaggle/working/outputs/final/mlp/*/seed-*/results.json'))]
cols = ['run', 'seed', 'best_epoch', 'threshold', 'params'] + [f'{s}_{m}' for s in ('train', 'val', 'test') for m in ('f1', 'precision', 'recall', 'pr_auc')]
pd.DataFrame(rows)[cols].round(4) if rows else "no finished seed yet"

,run,seed,best_epoch,threshold,params,train_f1,train_precision,train_recall,train_pr_auc,val_f1,val_precision,val_recall,val_pr_auc,test_f1,test_precision,test_recall,test_pr_auc
0,mlp_readout-full,42,12,0.3813,840705,0.5102,0.6704,0.4118,0.4517,0.5400,0.6832,0.4464,0.4731,0.4870,0.6008,0.4094,0.4221
1,mlp_readout-full,43,13,0.5367,840705,0.5190,0.7310,0.4023,0.4565,0.5385,0.7209,0.4298,0.4680,0.4938,0.6858,0.3858,0.4191
2,mlp_readout-full,44,25,0.6162,840705,0.5506,0.6903,0.4580,0.5185,0.5333,0.6603,0.4473,0.4695,0.4809,0.5879,0.4068,0.4199
3,mlp_readout-full,45,18,0.6402,840705,0.5404,0.7904,0.4105,0.4902,0.5432,0.7464,0.4270,0.4750,0.4928,0.7143,0.3762,0.4155
4,mlp_readout-full,46,12,0.5549,840705,0.5111,0.6865,0.4071,0.4483,0.5465,0.6707,0.4612,0.4777,0.5000,0.6553,0.4042,0.4334


In [7]:
# A small archive for download: everything except the checkpoints and predictions (the Output tab has the full folder).
!cd /kaggle/working && rm -f results_small.zip && zip -qr results_small.zip outputs -x "*.pt" "*predictions.csv" && ls -lh results_small.zip

-rw-r--r-- 1 root root 1.1M Nov 15 03:13 results_small.zip


## After the run
- Output tab: download `results_small.zip` (search log, metrics, curves, histories) or the whole `outputs` folder (adds `predictions.csv` and `best.pt`).
- Unzip into `gnn_for_aml/outputs/` on the laptop; commit the small files (`predictions.csv` and `*.pt` are gitignored); mark the MLP row in `EXPERIMENTS.md`.